# Entrevistas Técnicas — Ejercicios Guiados con Teoría
Nivel: PhD | Actualizado: Septiembre 2026

Cada problema incluye: **Concepto → Complejidad → Solución → Tests**

---
## Ejercicio 1: Two Sum (LC #1)

**Concepto:** Hash Table para lookup O(1). En lugar de buscar el complemento con loop O(n), guárdalo en un dict.

**Complejidad:** O(n) tiempo, O(n) espacio

In [ ]:
def two_sum(nums: list[int], target: int) -> list[int]:
    vistos = {}
    for i, num in enumerate(nums):
        complemento = target - num
        if complemento in vistos:
            return [vistos[complemento], i]
        vistos[num] = i
    return []

assert two_sum([2,7,11,15], 9) == [0,1]
assert two_sum([3,2,4], 6) == [1,2]
print("OK")

---
## Ejercicio 2: LRU Cache (LC #146)

**Concepto:** LRU (Least Recently Used) elimina el elemento menos usado cuando el cache está lleno. Se necesita O(1) para get y put.

**Implementación:** OrderedDict (Python) o HashMap + Doubly Linked List (entrevista).

In [ ]:
from collections import OrderedDict

class LRUCache:
    def __init__(self, capacity: int):
        self.cache = OrderedDict()
        self.capacity = capacity

    def get(self, key: int) -> int:
        if key not in self.cache:
            return -1
        self.cache.move_to_end(key)
        return self.cache[key]

    def put(self, key: int, value: int) -> None:
        if key in self.cache:
            self.cache.move_to_end(key)
        self.cache[key] = value
        if len(self.cache) > self.capacity:
            self.cache.popitem(last=False)

cache = LRUCache(2)
cache.put(1, 1); cache.put(2, 2)
assert cache.get(1) == 1
cache.put(3, 3)
assert cache.get(2) == -1
print("OK")

---
## Ejercicio 3: Valid Parentheses (LC #20)

**Concepto:** Stack (LIFO). Los paréntesis de apertura se apilan. Al encontrar uno de cierre, debe coincidir con el tope del stack.

**Complejidad:** O(n) tiempo, O(n) espacio

In [ ]:
def is_valid(s: str) -> bool:
    stack = []
    mapping = {')': '(', ']': '[', '}': '{'}
    for char in s:
        if char in mapping:
            top = stack.pop() if stack else '#'
            if mapping[char] != top:
                return False
        else:
            stack.append(char)
    return not stack

assert is_valid("()[]{}") == True
assert is_valid("(]") == False
assert is_valid("{[]}") == True
print("OK")

---
## Ejercicio 4: Coin Change (LC #322) — Dynamic Programming

**Concepto:** DP Bottom-Up. Para cada monto m, la mínima monedas es: `dp[m] = min(dp[m - coin] + 1)` para cada moneda.

**Complejidad:** O(monto × num_monedas)

In [ ]:
def coin_change(coins: list[int], amount: int) -> int:
    dp = [float('inf')] * (amount + 1)
    dp[0] = 0
    for i in range(1, amount + 1):
        for coin in coins:
            if coin <= i:
                dp[i] = min(dp[i], dp[i - coin] + 1)
    return dp[amount] if dp[amount] != float('inf') else -1

assert coin_change([1,5,10,25], 30) == 2
assert coin_change([2], 3) == -1
print("OK")

---
## Ejercicio 5: Binary Tree Level Order (LC #102) — BFS

**Concepto:** BFS (Breadth-First Search) usa una cola para recorrer nivel por nivel. Cada nivel es una lista.

**Complejidad:** O(n)

In [ ]:
from collections import deque

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def level_order(root: TreeNode) -> list[list[int]]:
    if not root:
        return []
    result = []
    queue = deque([root])
    while queue:
        level = []
        for _ in range(len(queue)):
            node = queue.popleft()
            level.append(node.val)
            if node.left: queue.append(node.left)
            if node.right: queue.append(node.right)
        result.append(level)
    return result

# Test
root = TreeNode(3, TreeNode(9), TreeNode(20, TreeNode(15), TreeNode(7)))
assert level_order(root) == [[3], [9, 20], [15, 7]]
print("OK")

---
## Ejercicio 6: Course Schedule (LC #207) — Topological Sort

**Concepto:** Si los prerrequisitos forman un ciclo, es imposible terminar. Topological sort (Kahn's algorithm) detecta ciclos.

**Complejidad:** O(V + E)

In [ ]:
from collections import deque

def can_finish(num_courses: int, prerequisites: list[list[int]]) -> bool:
    in_degree = [0] * num_courses
    adj = [[] for _ in range(num_courses)]
    for dest, src in prerequisites:
        adj[src].append(dest)
        in_degree[dest] += 1
    queue = deque([i for i in range(num_courses) if in_degree[i] == 0])
    count = 0
    while queue:
        node = queue.popleft()
        count += 1
        for neighbor in adj[node]:
            in_degree[neighbor] -= 1
            if in_degree[neighbor] == 0:
                queue.append(neighbor)
    return count == num_courses

assert can_finish(2, [[1,0]]) == True
assert can_finish(2, [[1,0],[0,1]]) == False
print("OK")

---
## Ejercicio 7: Merge K Sorted Lists (LC #23) — Heap

**Concepto:** Min-heap (priority queue) para efficientemente encontrar el menor elemento entre K listas.

**Complejidad:** O(N log K) donde N es total de elementos

In [ ]:
import heapq

class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def merge_k_lists(lists: list) -> ListNode:
    heap = []
    for i, node in enumerate(lists):
        if node:
            heapq.heappush(heap, (node.val, i, node))
    dummy = ListNode(0)
    current = dummy
    while heap:
        val, i, node = heapq.heappop(heap)
        current.next = node
        current = current.next
        if node.next:
            heapq.heappush(heap, (node.next.val, i, node.next))
    return dummy.next

---
## Ejercicio 8: Rate Limiter — System Design

**Concepto:** Limitar peticiones por usuario/ventana de tiempo. 3 algoritmos principales:
1. **Fixed Window:** Cuenta peticiones en ventanas fijas (simple pero causa bursts)
2. **Sliding Window:** Ventana deslizante (más suave)
3. **Token Bucket:** Tokens que se reponen a velocidad constante (flexible)

In [ ]:
import time
from collections import deque

class SlidingWindowRateLimiter:
    def __init__(self, max_requests: int, window_seconds: float):
        self.max_requests = max_requests
        self.window_seconds = window_seconds
        self.timestamps = deque()

    def allow(self) -> bool:
        now = time.time()
        # Eliminar timestamps fuera de ventana
        while self.timestamps and self.timestamps[0] < now - self.window_seconds:
            self.timestamps.popleft()
        if len(self.timestamps) < self.max_requests:
            self.timestamps.append(now)
            return True
        return False

# Test
limiter = SlidingWindowRateLimiter(3, 1.0)
assert limiter.allow() == True
assert limiter.allow() == True
assert limiter.allow() == True
assert limiter.allow() == False  # Límite alcanzado
print("OK")

---
## Ejercicio 9: K-Means desde Cero — ML Interview

**Concepto:** K-Means: (1) Inicializar K centroides, (2) Asignar puntos al centroide más cercano, (3) Recalcular centroides, (4) Repetir hasta convergencia.

**K-Means++:** Mejor inicialización — elige centroides que estén lejos entre sí.

In [ ]:
import numpy as np

def kmeans_pp_init(X, k):
    """K-Means++ initialization"""
    n_samples = X.shape[0]
    centroids = [X[np.random.randint(n_samples)]]
    for _ in range(1, k):
        dists = np.min([np.sum((X - c)**2, axis=1) for c in centroids], axis=0)
        probs = dists / dists.sum()
        centroids.append(X[np.random.choice(n_samples, p=probs)])
    return np.array(centroids)

def kmeans(X, k, max_iter=100):
    centroids = kmeans_pp_init(X, k)
    for _ in range(max_iter):
        # Asignar
        dists = np.array([np.sum((X - c)**2, axis=1) for c in centroids])
        labels = np.argmin(dists, axis=0)
        # Actualizar centroides
        new_centroids = np.array([X[labels == i].mean(axis=0) if np.sum(labels == i) > 0 else centroids[i] for i in range(k)])
        if np.allclose(centroids, new_centroids):
            break
        centroids = new_centroids
    return labels, centroids

# Test con blobs
from sklearn.datasets import make_blobs
X, _ = make_blobs(n_samples=300, centers=3, random_state=42)
labels, centroids = kmeans(X, 3)
print(f"Clusters encontrados: {len(np.unique(labels))}")

---
## Ejercicio 10: System Design — Diseña un Sistema de Recomendaciones

**Concepto:** Los sistemas de recomendación tienen 3 capas:
1. **Candidate Generation:** Reducir millones de items a cientos (collaborative filtering, content-based)
2. **Scoring/Ranking:** Reordenar candidatos (ML model con features de usuario+item)
3. **Re-ranking:** Aplicar reglas de negocio (diversidad, frescura, stock)

**Escala:** Si tienes 100M usuarios y 1M items, necesitas:
- Pre-computación batch (candidatos)
- Serving en tiempo real (ranking)
- Cache (Redis) para respuestas rápidas

In [ ]:
# EJERCICIO 10: Diseña por escrito un sistema de recomendaciones
# Incluye: arquitectura, componentes, flujo de datos, escalabilidad
# No requiere código — es un ejercicio de diseño